# Team AEC Possession Shapes

Find a team's highest `aec_per_throw` scoring possessions and compare them with the middle of the filtered efficient-possession set.

In [2]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from ufa_aec_possessions import (
    build_aec_possession_sets,
    build_scoring_possessions,
    fetch_shownspace_season_throws,
    plot_possession_path,
    plot_representative_paths,
    select_top_aec_possessions_by_team,
)

In [3]:
TEAM_ID = 'spiders'
SEASON = 2026
MAX_GAMES = None
METRIC = 'aec_per_throw'
TOP_N = 5
MIDDLE_COUNT = 5

## Load Shown Space Throws

In [4]:
games, throws = fetch_shownspace_season_throws(
    season=SEASON,
    team_id=TEAM_ID,
    max_games=MAX_GAMES,
)
possessions, paths = build_scoring_possessions(throws, team_id=TEAM_ID)

print(f'Games loaded: {len(games):,}')
print(f'Throws loaded: {len(throws):,}')
print(f'Scoring possessions found: {len(possessions):,}')

Games loaded: 10
Throws loaded: 5,671
Scoring possessions found: 234


## Build Highest and Middle AEC Sets

Defaults: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [5]:
sets = build_aec_possession_sets(
    possessions,
    paths,
    top_n=TOP_N,
    middle_count=MIDDLE_COUNT,
    metric=METRIC,
)
filtered_possessions, filtered_paths = sets['filtered']
highest_possessions, highest_paths = sets['highest']
middle_possessions, middle_paths = sets['middle']

print(f'Filtered analysis possessions: {len(filtered_possessions):,}')

Filtered analysis possessions: 78


In [6]:
summary_columns = [
    'possession_id', 'GameID', 'line_type', 'start_y', 'end_y',
    'field_progress', 'throw_count', 'huck_count', 'total_aec', METRIC,
    'shape_width', 'shape_middle_third_share', 'shape_sideline_share',
]
highest_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-04-26-ORE-OAK|3|11|5|True,2026-04-26-ORE-OAK,o_line,40.00,105.54,65.54,4,0,1.103261,0.275815,19.68,0.750000,0.250000
1,2026-06-12-MIN-OAK|4|6|3|True,2026-06-12-MIN-OAK,o_line,43.81,102.54,58.73,4,0,1.000814,0.250204,9.45,1.000000,0.000000
2,2026-06-26-OAK-COL|3|3|4|False,2026-06-26-OAK-COL,o_line,40.00,105.90,65.90,4,0,0.999723,0.249931,18.77,0.125000,0.375000
3,2026-06-27-OAK-SLC|2|7|5|False,2026-06-27-OAK-SLC,o_line,37.87,105.54,67.67,5,0,1.003590,0.200718,18.64,0.300000,0.200000
4,2026-05-09-OAK-SEA|2|1|1|False,2026-05-09-OAK-SEA,o_line,14.18,106.90,92.72,6,0,1.070602,0.178434,25.37,0.416667,0.166667


In [7]:
middle_possessions.reindex(columns=summary_columns)

,possession_id,GameID,line_type,start_y,end_y,field_progress,throw_count,huck_count,total_aec,aec_per_throw,shape_width,shape_middle_third_share,shape_sideline_share
0,2026-05-09-OAK-SEA|4|5|3|False,2026-05-09-OAK-SEA,o_line,22.77,104.52,81.75,10,0,0.968672,0.096867,27.28,0.450000,0.150000
1,2026-05-02-SLC-OAK|4|4|1|True,2026-05-02-SLC-OAK,o_line,40.00,106.91,66.91,10,0,0.988247,0.098825,24.21,0.800000,0.000000
2,2026-06-27-OAK-SLC|3|7|1|False,2026-06-27-OAK-SLC,o_line,40.38,115.48,75.10,10,0,1.000831,0.100083,45.33,0.400000,0.250000
3,2026-05-30-SD-OAK|3|8|1|True,2026-05-30-SD-OAK,o_line,11.35,102.45,91.10,10,0,1.009702,0.100970,37.59,0.200000,0.150000
4,2026-05-10-OAK-ORE|1|12|2|False,2026-05-10-OAK-ORE,o_line,19.35,114.25,94.90,11,0,1.115989,0.101454,43.47,0.318182,0.318182


## Overlay Highest Possessions

In [11]:
highest_lookup = {path['possession_id'].iloc[0]: path for path in highest_paths}
highest_labeled_paths = {
    f'top {rank + 1}: {row[METRIC]:.3f}': highest_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(highest_possessions.iterrows())
    if row['possession_id'] in highest_lookup
}

if highest_labeled_paths:
    fig = plot_representative_paths(
        highest_labeled_paths,
        title=f'{TEAM_ID.title()} highest {len(highest_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## Overlay Middle Possessions

In [12]:
middle_lookup = {path['possession_id'].iloc[0]: path for path in middle_paths}
middle_labeled_paths = {
    f'middle {rank + 1}: {row[METRIC]:.3f}': middle_lookup[row['possession_id']]
    for rank, (_, row) in enumerate(middle_possessions.iterrows())
    if row['possession_id'] in middle_lookup
}

if middle_labeled_paths:
    fig = plot_representative_paths(
        middle_labeled_paths,
        title=f'{TEAM_ID.title()} middle {len(middle_labeled_paths)} non-huck long-field scoring possessions',
    )
    fig.show()

## League-Wide Top 5 By Team

Use the same default filter as above: O-line scoring possessions, long-field starts, hucks excluded, ranked by `aec_per_throw`.

In [ ]:
league_games, league_throws = fetch_shownspace_season_throws(
    season=SEASON,
    max_games=MAX_GAMES,
)
league_possessions, league_paths = build_scoring_possessions(league_throws)

league_top_possessions, league_top_paths_by_team = select_top_aec_possessions_by_team(
    league_possessions,
    league_paths,
    metric=METRIC,
    n=TOP_N,
)

print(f'League games loaded: {len(league_games):,}')
print(f'League throws loaded: {len(league_throws):,}')
print(f'League scoring possessions found: {len(league_possessions):,}')
print(f'Teams with qualifying possessions: {league_top_possessions["team_id"].nunique() if not league_top_possessions.empty else 0:,}')

In [ ]:
league_summary_columns = [
    'team_id', 'team_rank', 'possession_id', 'GameID', 'line_type',
    'start_y', 'end_y', 'field_progress', 'throw_count', 'huck_count',
    'total_aec', METRIC, 'shape_width', 'shape_middle_third_share',
    'shape_sideline_share',
]
league_top_possessions.reindex(columns=league_summary_columns)

## Overlay Each Team's Top 5

In [ ]:
from IPython.display import display

LEAGUE_TEAMS_TO_PLOT = sorted(league_top_possessions['team_id'].dropna().unique())

for league_team_id in LEAGUE_TEAMS_TO_PLOT:
    team_possessions = league_top_possessions[
        league_top_possessions['team_id'].eq(league_team_id)
    ].reset_index(drop=True)
    team_paths = league_top_paths_by_team.get(str(league_team_id), [])
    team_lookup = {path['possession_id'].iloc[0]: path for path in team_paths}
    team_labeled_paths = {
        f'top {rank + 1}: {row[METRIC]:.3f}': team_lookup[row['possession_id']]
        for rank, (_, row) in enumerate(team_possessions.iterrows())
        if row['possession_id'] in team_lookup
    }
    if not team_labeled_paths:
        continue
    fig = plot_representative_paths(
        team_labeled_paths,
        title=f'{str(league_team_id).title()} top {len(team_labeled_paths)} non-huck long-field scoring possessions',
    )
    display(fig)